# 0929 20일차

## 1. 모델별 차원

| | DNN | RNN | CNN |
|---|---|---|---|
| 데이터 구조 | 2차원 | 3차원 | 4차원 |
| `input_shape` | 1개 | 2개 | 3개 |
| output | 2차원 | 2차원 (§2) | 4차원 |

1. DNN : `(N, feature)` → `(N, units)`
2. RNN : `(N, timesteps, feature)` → `(N, units)`
3. CNN : `(N, 세로, 가로, 채널)` → `(N, 세로, 가로, filters)`

- `input_shape`은 데이터 구조보다 하나 적음. 맨 앞 장수를 뺌
- CNN만 4차원으로 나와 `Flatten`이나 `GAP`이 필요함

## 2. return_sequences

RNN이 마지막 칸 결과만 내보낼지, 칸마다 내보낼지 정하는 인자

![기본값은 마지막 칸의 결과 하나만 내보내 2차원이 되고, return_sequences를 켜면 칸마다 결과가 나와 timesteps 축이 남은 3차원이 된다. 두 층을 쌓으면 1층의 칸별 출력이 2층의 입력 시퀀스가 되고, 마지막 층만 기본값으로 두어 Dense에 연결한다](assets/rnn-stack.svg)

RNN은 3차원을 받는데 기본 출력이 2차원이라, 그대로는 RNN을 쌓을 수 없음

```python
model.add(LSTM(10, input_shape=(3, 1), return_sequences=True))   # (N, 3, 10)
model.add(LSTM(5))                                               # (N, 5)
model.add(Dense(1))
```

1. `False` (기본) : 마지막 칸만. `(N, units)`
2. `True` : 칸마다. `(N, timesteps, units)`

- 마지막 RNN 층에서는 끔. `Dense`가 2차원을 받기 때문
- 빼먹고 쌓으면 `expected ndim=3, found ndim=2`

### 2-1. 쌓았을 때의 모양

| | timesteps | feature |
|---|---|---|
| 입력 | 3 | 1 |
| 1층 `LSTM(10)` 출력 | 3 | 10 |
| 2층 `LSTM(5)` 출력 | — | 5 |

```
1층  4 × 10 × (1 + 10 + 1) = 480
2층  4 ×  5 × (10 + 5 + 1) = 320
```

- 앞 층의 `units`가 다음 층의 `feature`가 됨
- 쌓는 것은 timesteps가 길고 패턴이 겹칠 때 의미가 있음

### 2-2. 영향 범위

![Dense는 아래 노드 셋이 위 노드 셋 전부와 이어져 연결이 아홉 개다. RNN을 쌓으면 같은 칸끼리만 세로로 직접 이어지고, 앞 칸은 2층의 가로 순환을 거쳐 뒤 칸 계산에까지 반영된다. 입력 하나를 바꿔 측정하면 앞 입력일수록 영향 범위가 넓고, 뒤 입력은 이미 지나간 칸에 영향을 주지 못해 삼각형 모양이 된다](assets/rnn-influence.svg)

![1층의 각 칸이 2층 어느 칸까지 반영되는지 한 칸씩 따라가는 애니메이션. 앞 칸일수록 반영되는 범위가 넓고 뒤 칸일수록 좁아진다](assets/rnn-influence.gif)

1. 같은 칸 안 : `Dense`처럼 전부 연결됨
2. 칸 사이 : 앞에서 뒤로만. 2층의 순환을 거쳐 반영됨

- 1층 h₁은 2층 h₁에만 직접 이어지지만 순환을 거쳐 세 칸 모두에 반영됨
- 1층 h₃은 이미 계산이 끝난 2층 h₁, h₂에 반영되지 않음
- `Dense`는 꽉 찬 사각형, RNN 쌓기는 삼각형

## 3. drop

지정한 행이나 열을 뺀 새 DataFrame을 돌려줌

```python
data.drop(data.index[-144:], axis=0)     # 행. axis 기본값이 0
data.drop('wd (deg)', axis=1)            # 열
```

- `.size`는 행 × 열 전체 칸 수. 행 개수는 `len()`이나 `.shape[0]`

### 3-1. 라벨과 위치

`drop`은 라벨(이름)만 받음. 위치로 자르려면 `iloc`

| 쓰는 법 | 받는 것 | 인덱스가 날짜여도 |
|---|---|---|
| `drop([5, 6])` | `5`, `6`이라는 이름의 행 | 안 됨 |
| `drop(df.index[-144:])` | 뒤 144개의 이름 | 됨 |
| `df.iloc[:-144]` | 위치 | 됨 |

- 인덱스가 `0, 1, 2...`면 이름과 위치가 같아서 숫자로도 통함
- `df.index[...]`가 위치를 이름으로 바꿔 줌

### 3-2. index와 columns

`index`는 행 이름, `columns`는 열 이름. 지우려는 축에서 뽑음

```python
x = data.drop(data.columns[-1], axis=1)     # 마지막 열 빼기
y = data.drop(data.columns[:-1], axis=1)    # 마지막 열만 남기기
```

- `axis=1`에 `data.index`를 넘기면 `KeyError`
- `data.iloc[-1]`은 이름이 아니라 값이라 `drop`에 못 넘김
- 대괄호에 이름을 넣으면 열, 범위를 넣으면 행이 나옴

#### cf) drop은 원본을 바꾸지 않음

1. `r = d.drop(...)` : 원본 그대로
2. `d.drop(..., inplace=True)` : 원본이 바뀜
3. `d = d.drop(...)` : 변수가 새것을 가리켜 원본을 놓침

- 자른 구간을 나중에 쓰려면 먼저 다른 이름으로 빼둠
- `iloc` 슬라이스 값을 고칠 때는 `.copy()`. 안 하면 `SettingWithCopyWarning`

## 4. 실제 데이터로 RNN 실습

jena 기상 데이터로 12월 31일 하루치 풍향 144개를 맞히는 문제

1. 42만 행, 컬럼 14개. 10분 간격이라 하루가 144행
2. y는 `wd (deg)`, 나머지 13개가 x

- 창 끝에서 144스텝 뒤를 맞히므로 x와 y를 144칸 어긋나게 자름

### 4-1. 순서

1. 읽기 : `index_col=0`으로 날짜를 인덱스로
2. 정답 빼두기 : 뒤 144개의 `wd`를 다른 이름으로 저장
3. x, y 자르기 : x는 `[:-288]`, y는 `[144:-144]`
4. 스케일링 : 창을 만들기 전 2차원일 때
5. 창 만들기 : `(N, 144, 13)`
6. 모델 : `LSTM(input_shape=(144, 13))` 뒤에 `Dense`
7. 예측 : 287행을 창 144개로 만들고 같은 스케일러로 `transform`

- 2번을 건너뛰면 3번에서 `data`를 덮어써 정답을 꺼낼 수 없음
- 4·5번을 바꾸면 3차원이라 스케일러가 못 받음

### 4-2. 학습에서 빼는 구간

정답 144개와 그 앞 144개를 함께 뺌

```python
x_data = data[:-288]        # 정답 144 + 앞당긴 144
y_data = data[144:-144]     # 144칸 민 것
```

```
0 ─────────── 420262 │ 420263 ─ 420406 │ 420407 ─ 420550
     학습용 x         │   예측 입력      │     정답
```

- 앞 144행의 정답이 정답 구간 안에 있어서 학습에 쓰면 누수 (day05 §2 cf)
- 가운데 144행은 예측 입력으로 씀

### 4-3. 예측 입력 창

정답 144개를 맞히려면 창이 144개 필요함

```
행 수 = 창 길이 + 창 개수 - 1 = 144 + 144 - 1 = 287
```

- day19 §1-2의 `len - size + 1`을 거꾸로 푼 것
- 144행만 넣으면 창이 1개라 예측도 1건

#### cf) 스케일링은 창을 만들기 전에

`MinMaxScaler`는 2차원만 받음

```python
x_data = scaler.fit_transform(x_data)   # (420263, 13) 2차원일 때
x = split(x_data, size)                 # 그다음 창 만들기
```

- jena는 기압이 1000대라 스케일을 안 맞추면 다른 컬럼이 묻힘

#### cf) 창을 만들면 데이터가 창 길이배

| | 크기 |
|---|---|
| 원본 `(420263, 13)` | 42 MB |
| 창 `(420120, 144, 13)` | 5.86 GB |

- VRAM을 넘으면 `Dst tensor is not initialized`
- 창을 띄엄띄엄 만들거나 `float32`로 줄임
- `import tensorflow` 전에 할당 방식을 바꾸면 더 큰 것도 올라감

```python
import os
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
```